# Religion–Violence–Geography with SAEs
## I. Data Collection & Preliminary Analysis
---

This notebook contains the **data collection pipeline** and **early analysis steps** 
for the experiments described in:

**"Mechanistic Interpretability with SAEs: Probing Religion, Violence, and Geography in Large Language Models"**  
*(AEQUITAS Workshop @ ECAI 2025)*

It provides a reproducible workflow for:
- Step 1: Fetch SAE activations from the Neuronpedia API  
- Step 2: Extract logits and structure outputs  
- Step 3a: Identify duplicate latent features (shared across multiple queries)  
- Step 3b: Exploratory logit analysis of positive and negative strings (not included in the final paper)

These steps prepare and structure the activation data for subsequent analysis.  

**Note:**  
- For full **semantic probing** of activation texts (crime, geography, etc.), refer to the **Semantic Analysis notebook**.  
- For detailed **latent feature overlap analysis** (intra-group and inter-group overlaps), refer to the **Overlap Analysis notebook**.

---

## Requirements
- Python 3.9+  
- Jupyter Notebook  
- Dependencies: `pandas`, `numpy`, `matplotlib`, `seaborn`, `re`, `requests`  
- Access to [Neuronpedia API](https://www.neuronpedia.org)

---

## Notes
- This notebook is one of three notebooks which reproduce the **camera-ready experimental results**.  
- The notebook is modular: individual sections can be run independently.


---
## Setup: Import Dependencies

In [1]:
import pandas as pd
import json
import importlib

---
## 1. Data Collection
(steps 1-2)

### 1.1.1 Define Models and SAE Source Sets

We begin by specifying the models and SAE (Sparse Autoencoder) configurations that will be used in the analysis.  

- **`modelreleases`**: A list of the high-level LLMs included in this study  
  *(Gemma-2-2b, Gemma-2-9b, Gemma-2-9b-IT, GPT2-small, LLaMA3.1-8b)*.  

- **`model_source_sets`**: Each model can have multiple SAE checkpoints (trained at different positions or with different feature counts).  
  These source sets are tuples `(model_name, sae_name)` that link the model with the specific SAE variant to query.  

These mappings will be used downstream to fetch activations from Neuronpedia for all queries.

In [2]:
modelReleases = [
    "gemma-2-2b",
    "gemma-2-9b",
    "gemma-2-9b-it",
    "gpt2-small",
    "llama3.1-8b",
]

model_source_sets = [
    ("gpt2-small", "res-jb"),
    ("gpt2-small", "att-kk"),
    ("gemma-2-2b", "gemmascope-att-16k"),
    ("gemma-2-2b", "gemmascope-res-16k"), 
    ("gemma-2-2b", "gemmascope-res-65k"), 
    ("gemma-2-9b", "gemmascope-res-16k"), 
    ("gemma-2-9b-it", "gemmascope-res-16k"), 
    ("gemma-2-9b-it", "gemmascope-res-131k"),  
    ("llama3.1-8b","llamascope-res-32k"),
]

---
### 1.1.2 Load Religion and Violence Query Set

Here we load the curated set of **81 queries** from `queries7.json`.  
- Each query is a short declarative sentence (e.g., *“This is a church”*, *“This is a terrorist”*).  
- Queries span the five religions, violence-related terms, and neutral baseline terms.  
- These are the exact inputs described in the paper (Appendix tables).  

This list will be iterated over for all models and SAEs to retrieve latent feature activations.

In [3]:
with open('queries7.json', 'r') as file:
    queries = json.load(file)

print(f"Loaded {len(queries)} queries.")

Loaded 81 queries.


---
### 1.1.3 Fetch SAE Feature Activations via Neuronpedia API

We now call the helper script `step1_fetch_SAE_data_via_inference.py`, which handles querying the **Neuronpedia API**.  

- The script function `fetch_SAE_data_via_inference(...)` takes:  
  - `model_source_sets`: list of (model, SAE) pairs to analyze  
  - `queries`: the curated religion/violence prompts  
  - `base_dir`: directory where results will be saved (`json7/`)  

#### What happens internally:
1. For each `(model, SAE)` pair, the script sends every query string to the Neuronpedia API.  
2. The API returns the **top-20 latent features** activated for that query.  
3. Each feature is saved as a JSON file (with metadata: feature ID, model, SAE source, activation texts).  
4. Results are organized in a structured directory under `json7/` for later overlap and semantic analysis.

This step corresponds to the **data collection phase** in the paper:  
*"Query & Feature Collection – each religion-related query was passed to the SAE, and the top 20 activating latent features were retained."*

In [4]:
import step1_fetch_SAE_data_via_inference
importlib.reload(step1_fetch_SAE_data_via_inference)

from step1_fetch_SAE_data_via_inference import fetch_SAE_data_via_inference

fetch_SAE_data_via_inference(model_source_sets, queries, base_dir='json7')

Processing query 'This is a terrorist' for model 'gpt2-small' with source set 'res-jb'
Feature:  ('gpt2-small', 'res-jb', '12-res-jb', '20701')
model:  gpt2-small
https://www.neuronpedia.org/api/feature/gpt2-small/12-res-jb/20701
Data saved to json7/gpt2-small/res-jb/This is a terrorist/data_for_feature_20701.json
Feature:  ('gpt2-small', 'res-jb', '12-res-jb', '19540')
model:  gpt2-small
https://www.neuronpedia.org/api/feature/gpt2-small/12-res-jb/19540
Data saved to json7/gpt2-small/res-jb/This is a terrorist/data_for_feature_19540.json
Feature:  ('gpt2-small', 'res-jb', '12-res-jb', '15301')
model:  gpt2-small
https://www.neuronpedia.org/api/feature/gpt2-small/12-res-jb/15301
Data saved to json7/gpt2-small/res-jb/This is a terrorist/data_for_feature_15301.json
Feature:  ('gpt2-small', 'res-jb', '12-res-jb', '13887')
model:  gpt2-small
https://www.neuronpedia.org/api/feature/gpt2-small/12-res-jb/13887
Data saved to json7/gpt2-small/res-jb/This is a terrorist/data_for_feature_13887.js

---
### 1.2 Extract Logits for All Queries

This step processes the raw activations collected in **Step 1** and extracts the **logit-level outputs** 
associated with each query across models.

#### Purpose
- Parse the saved JSON files (from Step 1) containing latent feature activations.
- Collect logit information for each query, ensuring alignment across models.
- Store cleaned, structured outputs in the `json7/` directory for downstream analysis.

This prepares the dataset for overlap and semantic probing in later steps.

In [5]:
import step2_logits_extractor
importlib.reload(step2_logits_extractor)

from step2_logits_extractor import extract_logits_for_all_queries

extract_logits_for_all_queries(modelReleases, queries_file='queries7.json', main_dir='json7')

✅ This is a baptism => gemma-2-2b-gemmascope-att-16k
✅ This is a bible => gemma-2-2b-gemmascope-att-16k
✅ This is a christian => gemma-2-2b-gemmascope-att-16k
✅ This is a church => gemma-2-2b-gemmascope-att-16k
✅ This is a gospel => gemma-2-2b-gemmascope-att-16k
✅ This is a pope => gemma-2-2b-gemmascope-att-16k
✅ This is a sacrament => gemma-2-2b-gemmascope-att-16k
✅ This is christianity => gemma-2-2b-gemmascope-att-16k
✅ This is jesus => gemma-2-2b-gemmascope-att-16k
✅ This is a priest => gemma-2-2b-gemmascope-att-16k
✅ This is a pastor => gemma-2-2b-gemmascope-att-16k
✅ This is a crucifix => gemma-2-2b-gemmascope-att-16k
✅ This is communion => gemma-2-2b-gemmascope-att-16k
✅ This is a burka => gemma-2-2b-gemmascope-att-16k
✅ This is a hijab => gemma-2-2b-gemmascope-att-16k
✅ This is a mosque => gemma-2-2b-gemmascope-att-16k
✅ This is a muslim => gemma-2-2b-gemmascope-att-16k
✅ This is a quran => gemma-2-2b-gemmascope-att-16k
✅ This is allah => gemma-2-2b-gemmascope-att-16k
✅ This is 

✅ This is a gospel => gemma-2-9b-it-gemmascope-res-131k
✅ This is a grandma => gemma-2-9b-it-gemmascope-res-131k
✅ This is a grandpa => gemma-2-9b-it-gemmascope-res-131k
✅ This is a hijab => gemma-2-9b-it-gemmascope-res-131k
✅ This is a homemaker => gemma-2-9b-it-gemmascope-res-131k
✅ This is a homophob => gemma-2-9b-it-gemmascope-res-131k
✅ This is a husband => gemma-2-9b-it-gemmascope-res-131k
✅ This is a jew => gemma-2-9b-it-gemmascope-res-131k
✅ This is a king => gemma-2-9b-it-gemmascope-res-131k
✅ This is a kippah => gemma-2-9b-it-gemmascope-res-131k
✅ This is a lady => gemma-2-9b-it-gemmascope-res-131k
✅ This is a male => gemma-2-9b-it-gemmascope-res-131k
✅ This is a man => gemma-2-9b-it-gemmascope-res-131k
✅ This is a midwife => gemma-2-9b-it-gemmascope-res-131k
✅ This is a mosque => gemma-2-9b-it-gemmascope-res-131k
✅ This is a muslim => gemma-2-9b-it-gemmascope-res-131k
✅ This is a nurse => gemma-2-9b-it-gemmascope-res-131k
✅ This is a pope => gemma-2-9b-it-gemmascope-res-131k

✅ This is a receptionist => gpt2-small-res-jb
✅ This is a sacrament => gpt2-small-res-jb
✅ This is a son => gpt2-small-res-jb
✅ This is a synagogue => gpt2-small-res-jb
✅ This is a talmud => gpt2-small-res-jb
✅ This is a temple => gpt2-small-res-jb
✅ This is a terrorist => gpt2-small-res-jb
✅ This is a torah => gpt2-small-res-jb
✅ This is a wife => gpt2-small-res-jb
✅ This is a woman => gpt2-small-res-jb
✅ This is allah => gpt2-small-res-jb
✅ This is an assistant => gpt2-small-res-jb
✅ This is an engineer => gpt2-small-res-jb
✅ This is christianity => gpt2-small-res-jb
✅ This is cocaine => gpt2-small-res-jb
✅ this is female => gpt2-small-res-jb
✅ This is halal => gpt2-small-res-jb
✅ This is he => gpt2-small-res-jb
✅ This is islam => gpt2-small-res-jb
✅ This is jesus => gpt2-small-res-jb
✅ This is judaism => gpt2-small-res-jb
✅ This is kosher => gpt2-small-res-jb
✅ This is mecca => gpt2-small-res-jb
✅ This is naiveThis is money => gpt2-small-res-jb
✅ this is nose => gpt2-small-res-jb
✅ 

---
## 2. Data Analysis
(steps 3-4)

### 2.1 Duplicate Features

First, this step analyzes the collected activations to detect **which latent features fire for multiple queries**. 

#### Purpose
- For each model, scan the feature activations saved in `json7/`.  
- Detect latent features that appear across **different queries** (e.g., one feature firing for *“This is the Quran”*, *“This is a church”*, and *“This is a Bible”*).  
- Aggregate these shared activations into lists, grouped by feature.

In [6]:
import step3a_check_duplicates_inference
importlib.reload(step3a_check_duplicates_inference)

from step3a_check_duplicates_inference import check_duplicates_inference

check_duplicates_inference(modelReleases, main_dir='json7')

✅ 5 duplicate feature numbers found. gemma-2-2b => gemmascope-att-16k
✅ 25 duplicate feature numbers found. gemma-2-2b => gemmascope-res-16k
✅ 32 duplicate feature numbers found. gemma-2-2b => gemmascope-res-65k
✅ 23 duplicate feature numbers found. gemma-2-9b => gemmascope-res-16k
✅ 69 duplicate feature numbers found. gemma-2-9b-it => gemmascope-res-131k
✅ 75 duplicate feature numbers found. gemma-2-9b-it => gemmascope-res-16k
✅ 64 duplicate feature numbers found. gpt2-small => att-kk
✅ 94 duplicate feature numbers found. gpt2-small => res-jb
✅ 48 duplicate feature numbers found. llama3.1-8b => llamascope-res-32k


In [9]:
import step4_analyze_duplicate_features_inference
importlib.reload(step4_analyze_duplicate_features_inference)

from step4_analyze_duplicate_features_inference import analyze_duplicate_features

analyze_duplicate_features(modelReleases, main_dir='json7')

✅ Common features for gemma-2-2b-gemmascope-att-16k saved.
✅ Common features for gemma-2-2b-gemmascope-res-16k saved.
✅ Common features for gemma-2-2b-gemmascope-res-65k saved.
✅ Common features for gemma-2-9b-gemmascope-res-16k saved.
✅ Common features for gemma-2-9b-it-gemmascope-res-131k saved.
✅ Common features for gemma-2-9b-it-gemmascope-res-16k saved.
✅ Common features for gpt2-small-att-kk saved.
✅ Common features for gpt2-small-res-jb saved.
✅ Common features for llama3.1-8b-llamascope-res-32k saved.


=====

Then, we merge all per-model `common_features.csv` files into a single comparison table.

#### Output
- `combined_common_features.csv` (stored in `json7/`)  
  Each row = a query triplet (`Term1`, `Term2`, `Term3`)  
  Each column = one model-source pair, with the value = number of shared features.  

In [10]:
import step4b_combine_csv_inference
importlib.reload(step4b_combine_csv_inference)

from step4b_combine_csv_inference import combine_csv_files

combine_csv_files(modelReleases, main_dir='json7')

✅ Combined CSV saved to json7/combined_common_features.csv


---
### 2.2 Exploratory Logit Analysis (Negative & Positive Strings)

This step explores whether **positive and negative activation strings** (returned by the SAEs) 
show duplicate patterns across queries.

#### Purpose
- Load structured feature outputs (from Step 2).
- Scan the **negative strings** and **positive strings** associated with each feature.
- Detect repeated patterns across queries, writing results to summary JSON files.

#### Note
This analysis was **exploratory** and not included in the final version of the paper 
because it did not yield particularly descriptive or interpretable results.  
However, it remains part of the exploratory workflow and is documented here for completeness.

In [7]:
import step3b_analyze_negative_logits
import step3b_analyze_positive_logits
importlib.reload(step3b_analyze_negative_logits)
importlib.reload(step3b_analyze_positive_logits)

from step3b_analyze_negative_logits import analyze_negative_logits
from step3b_analyze_positive_logits import analyze_positive_logits

In [8]:
analyze_negative_logits(directory='json7')
analyze_positive_logits(directory='json7')

Negative string duplicate analysis written to duplicates_neg_count.json
Duplicate information saved to json7/duplicates_neg_info.json
Positive string duplicate analysis written to duplicates_pos_count.json
Duplicate information saved to json7/duplicates_pos_info.json


---
### Utility – Generate Feature Summary Index

This utility step creates a **summary index** of all latent features collected 
across models, SAEs, and queries.

This CSV is primarily a **sanity check / indexing tool**, not a core analysis.  
It allows quick inspection of which features were collected for which queries, and provides a compact dataset that can be used for debugging, coverage checks, or downstream exploratory analysis.

In [ ]:
import step2b_generate_feature_summary
importlib.reload(step2b_generate_feature_summary)

from step2b_generate_feature_summary import generate_feature_summary

generate_feature_summary(base_folder='json7')

Feature summary CSV saved to json7/feature_summary.csv
